<a href="https://colab.research.google.com/github/yashsati072-create/Movie-Recommendation-System/blob/main/Movie-Recommendation-System.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [99]:
import pandas as pd
import numpy as np
import os
import zipfile
os.environ['KAGGLE_API_TOKEN'] = ''
!pip install -q kaggle
!kaggle datasets download -d rounakbanik/the-movies-dataset

Dataset URL: https://www.kaggle.com/datasets/rounakbanik/the-movies-dataset
License(s): CC0-1.0
the-movies-dataset.zip: Skipping, found more recently modified local copy (use --force to force download)


In [100]:
with zipfile.ZipFile('the-movies-dataset.zip','r') as zip_ref:
  zip_ref.extractall('dataset')

base = "dataset"
credit = pd.read_csv(f"{base}/credits.csv")
keyword=pd.read_csv(f"{base}/keywords.csv")
link=pd.read_csv(f"{base}/links.csv")
link_small=pd.read_csv(f"{base}/links_small.csv")
movie_da=pd.read_csv(f"{base}/movies_metadata.csv")
rating=pd.read_csv(f"{base}/ratings.csv")
rating_sm=pd.read_csv(f"{base}/ratings_small.csv")

/tmp/ipykernel_2020/3571181893.py:9: DtypeWarning: Columns (10) have mixed types. Specify dtype option on import or set low_memory=False.
  movie_da=pd.read_csv(f"{base}/movies_metadata.csv")


# Data Inspection


In [101]:
credit.head(10)
print(f"Shape:{credit.shape}")
print(f"Columns:{list(credit.columns)}")
print('\n Missing Values:')
print(credit.isnull().sum())
print(f"Duplicate:{credit.duplicated().sum()}")



Shape:(45476, 3)
Columns:['cast', 'crew', 'id']

 Missing Values:
cast    0
crew    0
id      0
dtype: int64
Duplicate:37


In [102]:
credit=credit.drop_duplicates()

#Keyword Data Inespection

In [103]:
keyword.head(10)
print(f"Shape:{keyword.shape}")
print(f"Columns:{list(keyword.columns)}")
print('\n Missing Values:')
print(keyword.isnull().sum())
print(f"Duplicate:{keyword.duplicated().sum()}")
keyword=keyword.drop_duplicates()



Shape:(46419, 2)
Columns:['id', 'keywords']

 Missing Values:
id          0
keywords    0
dtype: int64
Duplicate:987


#Links Data inspection

In [104]:
print(f"Head of top 5",link.head(5))
print(f"Shape:{link.shape}"
)
print(f"Columns:{list(link.columns)}")
print('\n Missing Values:')
print(link.isnull().sum())
print(f"Duplicate:{link.duplicated().sum()})")

Head of top 5    movieId  imdbId   tmdbId
0        1  114709    862.0
1        2  113497   8844.0
2        3  113228  15602.0
3        4  114885  31357.0
4        5  113041  11862.0
Shape:(45843, 3)
Columns:['movieId', 'imdbId', 'tmdbId']

 Missing Values:
movieId      0
imdbId       0
tmdbId     219
dtype: int64
Duplicate:0)


In [105]:
link['tmdbId'].fillna(link["tmdbId"].median(),inplace=True)

/tmp/ipykernel_2020/1024811610.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  link['tmdbId'].fillna(link["tmdbId"].median(),inplace=True)


#Links small Data inspection


In [106]:
link_small.head(10)

,movieId,imdbId,tmdbId
0,1,114709,862.0
1,2,113497,8844.0
2,3,113228,15602.0
3,4,114885,31357.0
4,5,113041,11862.0
5,6,113277,949.0
6,7,114319,11860.0
7,8,112302,45325.0
8,9,114576,9091.0
9,10,113189,710.0


In [107]:
link_small.head(10)
print(f"Shape:{link_small.shape}"
)
print(f"Columns:{list(link_small.columns)}")
print('\n Missing Values:')
print(link_small.isnull().sum())
print(f"Duplicate:{link_small.duplicated().sum()})")

Shape:(9125, 3)
Columns:['movieId', 'imdbId', 'tmdbId']

 Missing Values:
movieId     0
imdbId      0
tmdbId     13
dtype: int64
Duplicate:0)


In [108]:

link_small['tmdbId'].fillna(link_small["tmdbId"].median(),inplace=True)

/tmp/ipykernel_2020/679080834.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  link_small['tmdbId'].fillna(link_small["tmdbId"].median(),inplace=True)


# movie metadata data inspection


In [109]:
movie_da.head(10)
print(f"Shape:{movie_da.shape}"
)
print(f"Columns:{list(movie_da.columns)}")
print('\n Missing Values:')
print(movie_da.isnull().sum())
print(f"Duplicate:{movie_da.duplicated().sum()})")

Shape:(45466, 24)
Columns:['adult', 'belongs_to_collection', 'budget', 'genres', 'homepage', 'id', 'imdb_id', 'original_language', 'original_title', 'overview', 'popularity', 'poster_path', 'production_companies', 'production_countries', 'release_date', 'revenue', 'runtime', 'spoken_languages', 'status', 'tagline', 'title', 'video', 'vote_average', 'vote_count']

 Missing Values:
adult                        0
belongs_to_collection    40972
budget                       0
genres                       0
homepage                 37684
id                           0
imdb_id                     17
original_language           11
original_title               0
overview                   954
popularity                   5
poster_path                386
production_companies         3
production_countries         3
release_date                87
revenue                      6
runtime                    263
spoken_languages             6
status                      87
tagline                  250

In [118]:
movie_da.tail()

,adult,belongs_to_collection,budget,genres,homepage,id,imdb_id,original_language,original_title,overview,...,release_date,revenue,runtime,spoken_languages,status,tagline,title,video,vote_average,vote_count
45461,False,No Collection,0,"[{'id': 18, 'name': 'Drama'}, {'id': 10751, 'n...",http://www.imdb.com/title/tt6209470/,439050,tt6209470,fa,رگ خواب,Rising and falling between a man and woman.,...,NaN,0.0,90.0,"[{'iso_639_1': 'fa', 'name': 'فارسی'}]",Released,Rising and falling between a man and woman,Subdue,False,4.0,1.0
45462,False,No Collection,0,"[{'id': 18, 'name': 'Drama'}]",No URL,111109,tt2028550,tl,Siglo ng Pagluluwal,An artist struggles to finish his work while a...,...,2011-11-17,0.0,360.0,"[{'iso_639_1': 'tl', 'name': ''}]",Released,NaN,Century of Birthing,False,9.0,3.0
45463,False,No Collection,0,"[{'id': 28, 'name': 'Action'}, {'id': 18, 'nam...",No URL,67758,tt0303758,en,Betrayal,"When one of her hits goes wrong, a professiona...",...,2003-08-01,0.0,90.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,A deadly game of wits.,Betrayal,False,3.8,6.0
45464,False,No Collection,0,[],No URL,227506,tt0008536,en,Satana likuyushchiy,"In a small town live two brothers, one a minis...",...,1917-10-21,0.0,87.0,[],Released,NaN,Satan Triumphant,False,0.0,0.0
45465,False,No Collection,0,[],No URL,461257,tt6980792,en,Queerama,50 years after decriminalisation of homosexual...,...,2017-06-09,0.0,75.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,NaN,Queerama,False,0.0,0.0


In [111]:
# # Remove extra spaces
# movie_da['belongs_to_collection'] = movie_da['belongs_to_collection'].astype(str).str.strip()


In [119]:
movie_da=movie_da.drop_duplicates()
movie_da['belongs_to_collection'] = movie_da['belongs_to_collection'].fillna('No Collection')
movie_da['homepage'] = movie_da['homepage'].fillna('No URL')
movie_da['imdb_id']=movie_da['imdb_id'].fillna('no imdb_id')
movie_da['original_language'] = movie_da['original_language'].fillna('en')



movie_da.head()

,adult,belongs_to_collection,budget,genres,homepage,id,imdb_id,original_language,original_title,overview,...,release_date,revenue,runtime,spoken_languages,status,tagline,title,video,vote_average,vote_count
0,False,"{'id': 10194, 'name': 'Toy Story Collection', ...",30000000,"[{'id': 16, 'name': 'Animation'}, {'id': 35, '...",http://toystory.disney.com/toy-story,862,tt0114709,en,Toy Story,"Led by Woody, Andy's toys live happily in his ...",...,1995-10-30,373554033.0,81.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,NaN,Toy Story,False,7.7,5415.0
1,False,No Collection,65000000,"[{'id': 12, 'name': 'Adventure'}, {'id': 14, '...",No URL,8844,tt0113497,en,Jumanji,When siblings Judy and Peter discover an encha...,...,1995-12-15,262797249.0,104.0,"[{'iso_639_1': 'en', 'name': 'English'}, {'iso...",Released,Roll the dice and unleash the excitement!,Jumanji,False,6.9,2413.0
2,False,"{'id': 119050, 'name': 'Grumpy Old Men Collect...",0,"[{'id': 10749, 'name': 'Romance'}, {'id': 35, ...",No URL,15602,tt0113228,en,Grumpier Old Men,A family wedding reignites the ancient feud be...,...,1995-12-22,0.0,101.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Still Yelling. Still Fighting. Still Ready for...,Grumpier Old Men,False,6.5,92.0
3,False,No Collection,16000000,"[{'id': 35, 'name': 'Comedy'}, {'id': 18, 'nam...",No URL,31357,tt0114885,en,Waiting to Exhale,"Cheated on, mistreated and stepped on, the wom...",...,1995-12-22,81452156.0,127.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Friends are the people who let you be yourself...,Waiting to Exhale,False,6.1,34.0
4,False,"{'id': 96871, 'name': 'Father of the Bride Col...",0,"[{'id': 35, 'name': 'Comedy'}]",No URL,11862,tt0113041,en,Father of the Bride Part II,Just when George Banks has recovered from his ...,...,1995-02-10,76578911.0,106.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Just When His World Is Back To Normal... He's ...,Father of the Bride Part II,False,5.7,173.0


In [113]:
print(f"Shape:{rating.shape}")
print(f"Columns:{list(rating.columns)}")
print("\n Missing Values:")
print(rating.isnull().sum())
print(f"Duplicate:{rating.duplicated().sum()})")

Shape:(26024289, 4)
Columns:['userId', 'movieId', 'rating', 'timestamp']

 Missing Values:
userId       0
movieId      0
rating       0
timestamp    0
dtype: int64
Duplicate:0)


In [114]:
print(f"Shape:{rating_sm}")
print(f"Columns:{list(rating_sm)}")
print("\n Missing Values:")
print(rating_sm.isnull().sum())
print(f"Duplicate:{rating_sm.duplicated()})")

Shape:        userId  movieId  rating   timestamp
0            1       31     2.5  1260759144
1            1     1029     3.0  1260759179
2            1     1061     3.0  1260759182
3            1     1129     2.0  1260759185
4            1     1172     4.0  1260759205
...        ...      ...     ...         ...
99999      671     6268     2.5  1065579370
100000     671     6269     4.0  1065149201
100001     671     6365     4.0  1070940363
100002     671     6385     2.5  1070979663
100003     671     6565     3.5  1074784724

[100004 rows x 4 columns]
Columns:['userId', 'movieId', 'rating', 'timestamp']

 Missing Values:
userId       0
movieId      0
rating       0
timestamp    0
dtype: int64
Duplicate:0         False
1         False
2         False
3         False
4         False
          ...  
99999     False
100000    False
100001    False
100002    False
100003    False
Length: 100004, dtype: bool)
